In [4]:
import pandas as pd
import numpy as np
from sklearn.metrics import roc_auc_score
from scipy.stats import ks_2samp
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

In [5]:
df = pd.read_csv('model_validation.csv', sep=';')

In [6]:
df.head()

,CC_ID,SCORING_ID,LOAN_ID,SCORE_BALL,APPLICATION_DATE,TARGET_30,TARGET_60,TARGET_90
0,1463,42,1107792,38,19.07.2022,0.0,0.0,0.0
1,1467,42,1107613,38,19.07.2022,0.0,0.0,0.0
2,1520,42,1108049,40,20.07.2022,0.0,0.0,0.0
3,1539,42,1108413,45,21.07.2022,1.0,0.0,0.0
4,1604,42,1110647,41,25.07.2022,0.0,0.0,0.0


In [7]:
df.shape

(80397, 8)

In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 80397 entries, 0 to 80396
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   CC_ID             80397 non-null  int64  
 1   SCORING_ID        80397 non-null  int64  
 2   LOAN_ID           80397 non-null  int64  
 3   SCORE_BALL        80397 non-null  int64  
 4   APPLICATION_DATE  80397 non-null  object 
 5   TARGET_30         77452 non-null  float64
 6   TARGET_60         77452 non-null  float64
 7   TARGET_90         77452 non-null  float64
dtypes: float64(3), int64(4), object(1)
memory usage: 4.9+ MB


In [9]:
df[(df['TARGET_90'].isnull()) & (df['SCORING_ID'] == 727)].head(15)

,CC_ID,SCORING_ID,LOAN_ID,SCORE_BALL,APPLICATION_DATE,TARGET_30,TARGET_60,TARGET_90
47222,780847,727,1528162,870,24.02.2025,NaN,NaN,NaN
47598,782382,727,1528337,870,25.02.2025,NaN,NaN,NaN
47765,783048,727,1528626,850,26.02.2025,NaN,NaN,NaN
47792,783188,727,1528850,870,26.02.2025,NaN,NaN,NaN
47947,783828,727,1529190,760,27.02.2025,NaN,NaN,NaN
48059,784157,727,1529240,870,27.02.2025,NaN,NaN,NaN
48218,784685,727,1529454,870,27.02.2025,NaN,NaN,NaN
48240,784770,727,1529512,750,27.02.2025,NaN,NaN,NaN
48252,784800,727,1529533,750,27.02.2025,NaN,NaN,NaN
48270,784897,727,1529606,870,27.02.2025,NaN,NaN,NaN


In [10]:
df['APPLICATION_DATE'] = pd.to_datetime(
    df['APPLICATION_DATE'],
    format='%d.%m.%Y'
)
df[df['TARGET_90'].isnull()]
df = df.dropna(subset=['TARGET_90'])

In [11]:
df.shape

(77452, 8)

In [8]:
df.groupby('SCORING_ID')['APPLICATION_DATE'].agg({min, max})

C:\Users\sh.nabiyev\AppData\Local\Temp\ipykernel_12048\985070687.py:1: FutureWarning: The provided callable <built-in function min> is currently using SeriesGroupBy.min. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "min" instead.
  df.groupby('SCORING_ID')['APPLICATION_DATE'].agg({min, max})
C:\Users\sh.nabiyev\AppData\Local\Temp\ipykernel_12048\985070687.py:1: FutureWarning: The provided callable <built-in function max> is currently using SeriesGroupBy.max. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "max" instead.
  df.groupby('SCORING_ID')['APPLICATION_DATE'].agg({min, max})


,min,max
SCORING_ID,,
42,2022-07-19,2026-03-10
122,2022-10-17,2026-03-13
184,2023-01-06,2026-02-18
343,2024-02-09,2025-11-19
504,2024-06-19,2026-03-27
727,2025-01-22,2026-06-08


In [9]:
from sklearn.metrics import roc_auc_score
from scipy.stats import ks_2samp
import pandas as pd

max_date = df['APPLICATION_DATE'].max()

summary = []

df = df.copy()
df['APPLICATION_DATE'] = pd.to_datetime(df['APPLICATION_DATE'])

targets = {
    'TARGET_30': 30,
    'TARGET_60': 60,
    'TARGET_90': 90
}

for scoring_id, df_model in df.groupby('SCORING_ID'):

    for target, maturity_days in targets.items():

        # Maturity filter
        cutoff_date = max_date - pd.Timedelta(days=maturity_days)

        tmp = (
            df_model.loc[
                df_model['APPLICATION_DATE'] <= cutoff_date,
                [target, 'SCORE_BALL']
            ]
            .dropna()
        )

        if len(tmp) < 50:
            continue

        if tmp[target].nunique() < 2:
            continue

        # AUC
        auc = roc_auc_score(
            tmp[target],
            -tmp['SCORE_BALL']
        )

        # GINI
        gini = 2 * auc - 1

        # KS
        good_scores = tmp.loc[
            tmp[target] == 0,
            'SCORE_BALL'
        ]

        bad_scores = tmp.loc[
            tmp[target] == 1,
            'SCORE_BALL'
        ]

        ks = ks_2samp(
            -good_scores,
            -bad_scores
        ).statistic

        summary.append({
            'SCORING_ID': scoring_id,
            'TARGET': target,
            'OBS': len(tmp),
            'BAD_RATE': round(tmp[target].mean() * 100, 2),
            'AUC': round(auc, 4),
            'GINI': round(gini, 4),
            'KS': round(ks, 4),
            'START_DATE': df_model['APPLICATION_DATE'].min().date(),
            'END_DATE': cutoff_date.date()
        })

summary_df = (
    pd.DataFrame(summary)
    .sort_values(['SCORING_ID', 'TARGET'])
    .reset_index(drop=True)
)

summary_df

,SCORING_ID,TARGET,OBS,BAD_RATE,AUC,GINI,KS,START_DATE,END_DATE
0,42,TARGET_30,2643,21.11,0.5459,0.0919,0.0776,2022-07-19,2026-05-09
1,42,TARGET_60,2643,6.92,0.5651,0.1303,0.1086,2022-07-19,2026-04-09
2,42,TARGET_90,2643,4.01,0.5420,0.0839,0.0783,2022-07-19,2026-03-10
3,122,TARGET_30,1272,30.82,0.5336,0.0672,0.1221,2022-10-17,2026-05-09
4,122,TARGET_60,1272,12.89,0.5398,0.0795,0.1599,2022-10-17,2026-04-09
5,122,TARGET_90,1271,8.18,0.5384,0.0768,0.1488,2022-10-17,2026-03-10
6,184,TARGET_30,6731,31.61,0.5679,0.1358,0.1171,2023-01-06,2026-05-09
7,184,TARGET_60,6731,15.72,0.5640,0.1279,0.1025,2023-01-06,2026-04-09
8,184,TARGET_90,6731,10.41,0.5639,0.1278,0.0969,2023-01-06,2026-03-10
9,343,TARGET_30,46728,68.95,0.5170,0.0340,0.0321,2024-02-09,2026-05-09


In [1]:
# SCORING_ID = 42
# Portfelning BAD RATE ko‘rsatkichlari TARGET_30 bo‘yicha 21.1%, TARGET_60 bo‘yicha 6.9% va TARGET_90 bo‘yicha 4.0% ni tashkil etadi. 
# Modelning AUC (0.54–0.57), GINI (0.08–0.13) va KS (0.08–0.11) qiymatlari past bo‘lib, modelning risk segmentlarini ajratish qobiliyati 
# cheklanganligini ko‘rsatadi.
# 
# SCORING_ID = 122
# BAD RATE darajasi TARGET_30 bo‘yicha 30.8% ga yetadi. AUC (0.53–0.54), GINI (0.07–0.08) va KS (0.12–0.16) qiymatlari modelning ajratuvchanlik
# darajasi past ekanligini ko‘rsatmoqda. 
# 
# SCORING_ID = 184
# Portfelning default darajasi nisbatan yuqori (TARGET_30 = 31.6%). AUC (0.56–0.57), GINI (0.13 atrofida) va KS (0.10–0.12) qiymatlari 
# modelning ma'lum darajada ajratuvchanlikka ega ekanligini ko‘rsatsa-da, natijalar kuchli diskriminatsion qobiliyat mavjudligini tasdiqlamaydi.
# 
# SCORING_ID = 343
# Portfel BAD RATE ko‘rsatkichlari juda yuqori (TARGET_30 = 69.0%, TARGET_60 = 48.1%, TARGET_90 = 37.8%). Shu bilan birga AUC (0.51–0.52), 
# GINI (0.02–0.03) va KS (0.02–0.03) qiymatlari juda past bo‘lib, model ballari va kuzatilgan risk darajasi o‘rtasidagi bog‘liqlik zaif 
# ekanligini ko‘rsatadi.
# 
# SCORING_ID = 504
# Portfelda BAD RATE yuqori bo‘lishiga qaramay (TARGET_30 = 45.1%), modelning AUC (0.60–0.62), GINI (0.21–0.24) va KS (0.16–0.18) ko‘rsatkichlari
# boshqa ayrim modellarga nisbatan yaxshiroq natijalarni namoyish etmoqda. Biroq ko‘rsatkichlar modelni yuqori sifatli deb baholash uchun 
# yetarli darajada emas.
# 
# SCORING_ID = 727
# Portfelning BAD RATE darajasi boshqa modellarga nisbatan ancha past (TARGET_30 = 7.3%, TARGET_60 = 2.1%, TARGET_90 = 1.2%). 
# AUC (0.72–0.73), GINI (0.44–0.46) va KS (0.33–0.37) qiymatlari model ballari bilan kuzatilgan risk o‘rtasida nisbatan kuchli bog‘liqlik 
# mavjudligini ko‘rsatadi. Tahlil qilingan modellar orasida ushbu model eng yaxshi diskriminatsion ko‘rsatkichlarni namoyish etgan.

In [10]:
df.groupby('SCORING_ID').agg(
    cnt=('SCORING_ID', 'count'),
    dr30=('TARGET_30', 'mean'),
    dr60=('TARGET_60', 'mean'),
    dr90=('TARGET_90', 'mean')
).reset_index()

,SCORING_ID,cnt,dr30,dr60,dr90
0,42,2643,0.211124,0.069240,0.040106
1,122,1272,0.308176,0.128931,0.081761
2,184,6731,0.316149,0.157183,0.104145
3,343,46728,0.689522,0.480954,0.378167
4,504,1118,0.450805,0.322004,0.263864
5,727,18960,0.071994,0.020939,0.011181


In [11]:
def make_decile(df_model, target, cutoff_date):

    tmp = df_model.loc[
        df_model['APPLICATION_DATE'] <= cutoff_date,
        [target, 'SCORE_BALL']
    ].dropna().copy()

    if len(tmp) < 100 or tmp[target].nunique() < 2:
        return None

    # =========================
    # DECILE (HIGH SCORE = LOW RISK assumption kept consistent)
    # =========================
    tmp['DECILE'] = pd.qcut(
        tmp['SCORE_BALL'],
        q=10,
        labels=False,
        duplicates='drop'
    ) + 1

    report = (
        tmp.groupby('DECILE')
        .agg(
            cnt=(target, 'count'),
            bads=(target, 'sum'),
            bad_rate=(target, 'mean'),
            min_score=('SCORE_BALL', 'min'),
            max_score=('SCORE_BALL', 'max')
        )
        .reset_index()
        .sort_values('DECILE')
    )

    report['bad_rate'] = (report['bad_rate'] * 100).round(2)

    report['cum_cnt'] = report['cnt'].cumsum()
    report['cum_bads'] = report['bads'].cumsum()

    return report

max_date = df['APPLICATION_DATE'].max()

targets = {
    'TARGET_30': 30,
    'TARGET_60': 60,
    'TARGET_90': 90
}

all_reports = {}

for scoring_id, df_model in df.groupby('SCORING_ID'):

    print("\n" + "=" * 100)
    print(f"MODEL: {scoring_id}")
    print("=" * 100)

    for target, maturity_days in targets.items():

        cutoff_date = max_date - pd.Timedelta(days=maturity_days)

        report = make_decile(df_model, target, cutoff_date)

        if report is None:
            print(f"\n{target}: insufficient data")
            continue

        all_reports[(scoring_id, target)] = report

        print("\n" + "-" * 100)
        print(f"MODEL = {scoring_id} | TARGET = {target}")
        print(f"PERIOD <= {cutoff_date.date()}")
        print("-" * 100)

        print(report)

        # =========================
        # MONOTONICITY CHECK (FIXED LOGIC)
        # =========================
        monotonic = report['bad_rate'].is_monotonic_decreasing
        violations = (report['bad_rate'].diff() > 0).sum()

        print(f"\nMonotonicity: {'PASS' if monotonic else 'FAIL'}")
        print(f"Violations: {violations}")

        print(
            f"Bad Rate Range: "
            f"{report['bad_rate'].max():.2f}% -> {report['bad_rate'].min():.2f}%"
        )


MODEL: 42

----------------------------------------------------------------------------------------------------
MODEL = 42 | TARGET = TARGET_30
PERIOD <= 2026-05-09
----------------------------------------------------------------------------------------------------
   DECILE  cnt  bads  bad_rate  min_score  max_score  cum_cnt  cum_bads
0       1  272  81.0     29.78       -150         34      272      81.0
1       2  350  73.0     20.86         35         38      622     154.0
2       3  211  52.0     24.64         39         40      833     206.0
3       4  282  60.0     21.28         41         42     1115     266.0
4       5  233  41.0     17.60         43         44     1348     307.0
5       6  269  47.0     17.47         45         48     1617     354.0
6       7  253  56.0     22.13         49         60     1870     410.0
7       8  280  66.0     23.57         61         67     2150     476.0
8       9  254  52.0     20.47         68         72     2404     528.0
9      10  23

In [ ]:
# Decile tahlili bo‘yicha xulosalar
# SCORING_ID = 42
# Bad Rate umumiy pasayish trendini ko‘rsatsa-da, barcha targetlar bo‘yicha 3–4 ta monotonicity buzilishi mavjud. 
# Yuqori score segmentlarida risk pasaymoqda, biroq ayrim o‘rta decilelarda riskning qayta oshishi kuzatiladi. 
# Model risklarni to‘liq tartiblashda barqaror emas.
# 
# SCORING_ID = 122
# Monotonicity eng zaif modellardan biri hisoblanadi. TARGET_30 va TARGET_60 da 4 ta, TARGET_90 da esa 6 ta buzilish aniqlangan. 
# Bad Rate decilelar bo‘yicha notekis harakat qilmoqda, bu esa score va faktik risk o‘rtasidagi bog‘liqlikning sust ekanligini ko‘rsatadi.
# 
# SCORING_ID = 184
# Bad Rate birinchi va oxirgi decilelar orasida sezilarli farq ko‘rsatmoqda, biroq barcha targetlar bo‘yicha 3 tadan monotonicity buzilishi mavjud. 
# Model ma'lum darajada risklarni segmentatsiya qilmoqda, lekin decilelar bo‘yicha risk tartibi to‘liq saqlanmagan.
# 
# SCORING_ID = 343
# Decilelar bo‘yicha Bad Rate deyarli o‘zgarmaydi (TARGET_30 da 63–71% oralig‘ida). Monotonicity buzilishlari bilan birga decilelar orasidagi 
# farq juda kichik. Model ballari mijoz riskini deyarli ajratib bera olmayapti.
# 
# SCORING_ID = 504
# Bad Rate pastdan yuqoriga umumiy kamayish trendiga ega, ammo barcha targetlar bo‘yicha 3 tadan monotonicity buzilishi mavjud. 
# Ayrim o‘rta decilelarda riskning kutilmagan oshishi kuzatiladi. 
# 
# SCORING_ID = 727
# Decile tahlili bo‘yicha eng yaxshi natijalarni namoyish etgan model. TARGET_30 da monotonicity to‘liq saqlangan va buzilishlar mavjud emas. 
# TARGET_60 va TARGET_90 da faqat 1 tadan kichik buzilish kuzatiladi. Bad Rate birinchi deciledan oxirgi decilegacha izchil kamayib boradi, bu 
# score va faktik risk o‘rtasidagi kuchli bog‘liqlikni tasdiqlaydi.

In [12]:
df.groupby('SCORING_ID')['APPLICATION_DATE'].agg({min, max})

C:\Users\sh.nabiyev\AppData\Local\Temp\ipykernel_12048\985070687.py:1: FutureWarning: The provided callable <built-in function min> is currently using SeriesGroupBy.min. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "min" instead.
  df.groupby('SCORING_ID')['APPLICATION_DATE'].agg({min, max})
C:\Users\sh.nabiyev\AppData\Local\Temp\ipykernel_12048\985070687.py:1: FutureWarning: The provided callable <built-in function max> is currently using SeriesGroupBy.max. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "max" instead.
  df.groupby('SCORING_ID')['APPLICATION_DATE'].agg({min, max})


,min,max
SCORING_ID,,
42,2022-07-19,2026-03-10
122,2022-10-17,2026-03-13
184,2023-01-06,2026-02-18
343,2024-02-09,2025-11-19
504,2024-06-19,2026-03-27
727,2025-01-22,2026-06-08


In [13]:
def calculate_psi(expected, actual, buckets=10):

    def get_bins(data):
        return np.linspace(data.min(), data.max(), buckets + 1)

    expected = np.array(expected)
    actual = np.array(actual)

    bin_edges = get_bins(expected)

    expected_counts, _ = np.histogram(expected, bins=bin_edges)
    actual_counts, _ = np.histogram(actual, bins=bin_edges)

    expected_perc = expected_counts / len(expected)
    actual_perc = actual_counts / len(actual)

    expected_perc = np.where(expected_perc == 0, 0.0001, expected_perc)
    actual_perc = np.where(actual_perc == 0, 0.0001, actual_perc)

    psi = np.sum((actual_perc - expected_perc) * np.log(actual_perc / expected_perc))

    return psi


# =========================
# SETTINGS
# =========================
max_date = df['APPLICATION_DATE'].max()

targets = {
    'TARGET_30': 30,
    'TARGET_60': 60,
    'TARGET_90': 90
}

summary = []


# =========================
# LOOP MODELS
# =========================
for scoring_id, df_model in df.groupby('SCORING_ID'):

    df_model = df_model.copy()
    df_model['APPLICATION_DATE'] = pd.to_datetime(df_model['APPLICATION_DATE'])

    # =========================
    # DEV / OOT SPLIT (GLOBAL PER MODEL)
    # =========================
    dates = np.sort(df_model['APPLICATION_DATE'].unique())

    if len(dates) < 10:
        continue

    split_date = dates[int(len(dates) * 0.7)]

    dev_df = df_model[df_model['APPLICATION_DATE'] <= split_date]
    oot_df = df_model[df_model['APPLICATION_DATE'] > split_date]

    for target, maturity_days in targets.items():

        # =========================
        # MATURITY FILTER (TARGET LEVEL ONLY)
        # =========================
        cutoff_date = max_date - pd.Timedelta(days=maturity_days)

        tmp = df_model.loc[
            df_model['APPLICATION_DATE'] <= cutoff_date,
            [target, 'SCORE_BALL']
        ].dropna()

        if len(tmp) < 100 or tmp[target].nunique() < 2:
            continue

        # =========================
        # AUC / GINI (CONSISTENT)
        # =========================
        auc = roc_auc_score(tmp[target], -tmp['SCORE_BALL'])
        gini = 2 * auc - 1

        # =========================
        # KS (FIXED DIRECTION)
        # =========================
        ks = ks_2samp(
            -tmp.loc[tmp[target] == 0, 'SCORE_BALL'],
            -tmp.loc[tmp[target] == 1, 'SCORE_BALL']
        ).statistic

        # =========================
        # MONOTONICITY (DECILE STABLE)
        # =========================
        tmp = tmp.copy()

        tmp['DECILE'] = pd.qcut(
            tmp['SCORE_BALL'],
            q=10,
            labels=False,
            duplicates='drop'
        ) + 1

        report = (
            tmp.groupby('DECILE')
            .agg(
                cnt=(target, 'count'),
                bads=(target, 'sum'),
                bad_rate=(target, 'mean')
            )
            .reset_index()
            .sort_values('DECILE')
        )

        report['bad_rate'] = report['bad_rate'] * 100

        monotonic = report['bad_rate'].is_monotonic_decreasing
        violations = (report['bad_rate'].diff() > 0).sum()

        # =========================
        # PSI (SAFE: SAME MODEL DISTRIBUTION ONLY)
        # =========================
        if len(dev_df) > 50 and len(oot_df) > 50:
            psi = calculate_psi(
                dev_df['SCORE_BALL'].dropna(),
                oot_df['SCORE_BALL'].dropna()
            )
        else:
            psi = np.nan

        # =========================
        # SAVE
        # =========================
        summary.append({
            'SCORING_ID': scoring_id,
            'TARGET': target,
            'OBS': len(tmp),
            'BAD_RATE': tmp[target].mean(),
            'AUC': auc,
            'GINI': gini,
            'KS': ks,
            'PSI': psi,
            'MONOTONIC': monotonic,
            'VIOLATIONS': violations,
            'DEV_START': dev_df['APPLICATION_DATE'].min(),
            'DEV_END': dev_df['APPLICATION_DATE'].max(),
            'OOT_START': oot_df['APPLICATION_DATE'].min(),
            'OOT_END': oot_df['APPLICATION_DATE'].max()
        })


summary_df = pd.DataFrame(summary)
summary_df

,SCORING_ID,TARGET,OBS,BAD_RATE,AUC,GINI,KS,PSI,MONOTONIC,VIOLATIONS,DEV_START,DEV_END,OOT_START,OOT_END
0,42,TARGET_30,2643,0.211124,0.545942,0.091884,0.077590,4.624665,False,3,2022-07-19,2024-03-06,2024-03-11,2026-03-10
1,42,TARGET_60,2643,0.069240,0.565131,0.130261,0.108563,4.624665,False,3,2022-07-19,2024-03-06,2024-03-11,2026-03-10
2,42,TARGET_90,2643,0.040106,0.541962,0.083924,0.078253,4.624665,False,4,2022-07-19,2024-03-06,2024-03-11,2026-03-10
3,122,TARGET_30,1272,0.308176,0.533602,0.067205,0.122124,0.488707,False,4,2022-10-17,2024-07-15,2024-07-16,2026-03-13
4,122,TARGET_60,1272,0.128931,0.539769,0.079538,0.159923,0.488707,False,4,2022-10-17,2024-07-15,2024-07-16,2026-03-13
5,122,TARGET_90,1271,0.081825,0.538424,0.076849,0.148820,0.488707,False,6,2022-10-17,2024-07-15,2024-07-16,2026-03-13
6,184,TARGET_30,6731,0.316149,0.567923,0.135846,0.117062,0.688074,False,3,2023-01-06,2025-03-03,2025-03-04,2026-02-18
7,184,TARGET_60,6731,0.157183,0.563953,0.127907,0.102537,0.688074,False,3,2023-01-06,2025-03-03,2025-03-04,2026-02-18
8,184,TARGET_90,6731,0.104145,0.563919,0.127838,0.096852,0.688074,False,3,2023-01-06,2025-03-03,2025-03-04,2026-02-18
9,343,TARGET_30,46728,0.689522,0.516976,0.033951,0.032126,2.504476,False,4,2024-02-09,2025-02-06,2025-02-07,2025-11-19


In [ ]:
# SCORING_ID = 42
# Modelning diskriminatsion ko‘rsatkichlari past (AUC 0.54–0.57, KS 0.08–0.11), decile tahlilida 3–4 ta monotonicity buzilishi mavjud. 
# PSI = 4.62 bo‘lib, development va OOT portfellar orasida juda katta taqsimot o‘zgarishi kuzatiladi. 
# Modelning ajratuvchanligi ham, stabiligi ham qoniqarsiz.
# 
# SCORING_ID = 122 
# AUC, GINI va KS qiymatlari past darajada. Decile tahlilida 4–6 ta monotonicity buzilishi aniqlangan. 
# PSI = 0.49 bo‘lib, model taqsimotida sezilarli drift mavjud. Modelning risklarni tartiblash qobiliyati va vaqt bo‘yicha barqarorligi cheklangan.
# 
# SCORING_ID = 184
# Model 42 va 122 ga nisbatan biroz yaxshiroq diskriminatsiya ko‘rsatmoqda, biroq GINI va KS hali ham past. 
# Barcha targetlar bo‘yicha 3 ta monotonicity buzilishi mavjud. PSI = 0.69 bo‘lib, sezilarli portfel o‘zgarishi kuzatiladi. 
# Modelning stabiligi qoniqarsiz baholanadi.
# 
# SCORING_ID = 343
# Modelning AUC (0.51–0.52), GINI (0.02–0.03) va KS (0.02–0.03) qiymatlari juda past. Decilelar bo‘yicha Bad Rate deyarli farqlanmaydi. 
# PSI = 2.50 bo‘lib, juda kuchli drift mavjud. Model risklarni ajratish va vaqt davomida barqaror ishlash nuqtai nazaridan eng zaif natijalarni ko‘rsatmoqda.
# 
# SCORING_ID = 504
# Modelning diskriminatsion ko‘rsatkichlari o‘rtacha darajada (AUC 0.60–0.62, KS 0.16–0.18). Decile tahlilida 3 ta monotonicity buzilishi mavjud. 
# PSI = 1.36 bo‘lib, development va OOT davrlari o‘rtasida katta taqsimot farqi mavjud. Modelning stabiligi past.
# 
# SCORING_ID = 727
# Tahlildagi eng yaxshi model. AUC (0.72–0.73), GINI (0.44–0.46) va KS (0.33–0.37) boshqa modellarga nisbatan ancha yuqori. 
# TARGET_30 bo‘yicha monotonicity to‘liq saqlangan, TARGET_60 va TARGET_90 da esa faqat bittadan buzilish mavjud. PSI = 0.21 bo‘lib, 
# model taqsimoti nisbatan barqaror. Diskriminatsiya va stabilik nuqtai nazaridan eng yaxshi natijalar ushbu modelda kuzatiladi.

In [14]:
def calculate_psi(expected, actual, buckets=10):

    expected = np.array(expected)
    actual = np.array(actual)

    # =========================
    # GLOBAL BIN BASED ON COMBINED DISTRIBUTION (FIX)
    # =========================
    combined = np.concatenate([expected, actual])
    bin_edges = np.linspace(combined.min(), combined.max(), buckets + 1)

    expected_counts, _ = np.histogram(expected, bins=bin_edges)
    actual_counts, _ = np.histogram(actual, bins=bin_edges)

    expected_perc = expected_counts / len(expected)
    actual_perc = actual_counts / len(actual)

    expected_perc = np.where(expected_perc == 0, 0.0001, expected_perc)
    actual_perc = np.where(actual_perc == 0, 0.0001, actual_perc)

    psi = np.sum((actual_perc - expected_perc) * np.log(actual_perc / expected_perc))

    return psi

all_psi_score = {}

for scoring_id, df_model in df.groupby('SCORING_ID'):

    df_model = df_model.copy()
    df_model['APPLICATION_DATE'] = pd.to_datetime(df_model['APPLICATION_DATE'])

    dates = np.sort(df_model['APPLICATION_DATE'].unique())

    if len(dates) < 10:
        continue

    split_date = dates[int(len(dates) * 0.7)]

    dev_df = df_model[df_model['APPLICATION_DATE'] <= split_date]
    oot_df = df_model[df_model['APPLICATION_DATE'] > split_date]

    if len(dev_df) < 100 or len(oot_df) < 100:
        continue

    psi = calculate_psi(
        dev_df['SCORE_BALL'].dropna(),
        oot_df['SCORE_BALL'].dropna()
    )

    all_psi_score[scoring_id] = psi

    print(f"\nMODEL {scoring_id} | SCORE PSI = {psi:.4f}")


MODEL 42 | SCORE PSI = 4.6247

MODEL 122 | SCORE PSI = 0.4582

MODEL 184 | SCORE PSI = 0.6881

MODEL 343 | SCORE PSI = 2.4028

MODEL 504 | SCORE PSI = 1.4819

MODEL 727 | SCORE PSI = 0.2142


In [15]:
all_psi_time = {}

for scoring_id, df_model in df.groupby('SCORING_ID'):

    df_model = df_model.copy()
    df_model['APPLICATION_DATE'] = pd.to_datetime(df_model['APPLICATION_DATE'])

    df_model['MONTH'] = df_model['APPLICATION_DATE'].dt.to_period('M')

    months = sorted(df_model['MONTH'].unique())

    results = []

    for i in range(1, len(months)):

        prev = df_model[df_model['MONTH'] == months[i-1]]['SCORE_BALL'].dropna()
        curr = df_model[df_model['MONTH'] == months[i]]['SCORE_BALL'].dropna()

        if len(prev) < 50 or len(curr) < 50:
            continue

        # =========================
        # FIX: GLOBAL BIN CONSISTENCY
        # =========================
        psi = calculate_psi(prev, curr)

        results.append({
            'MONTH': str(months[i]),
            'PSI': psi,
            'OBS_PREV': len(prev),
            'OBS_CURR': len(curr)
        })

    psi_df = pd.DataFrame(results)

    all_psi_time[scoring_id] = psi_df

    print("\n" + "="*60)
    print(f"MODEL {scoring_id} - TIME PSI")
    print("="*60)

    print(psi_df)


MODEL 42 - TIME PSI
     MONTH       PSI  OBS_PREV  OBS_CURR
0  2022-09  0.164437       166       255
1  2022-10  0.064444       255       257
2  2022-11  0.073052       257       387
3  2022-12  0.504938       387       342
4  2023-01  0.828048       342        87
5  2023-02  0.271650        87       109
6  2023-03  3.429782       109        82
7  2023-09  0.127056       275        61
8  2024-05  0.241962        64        76
9  2024-06  0.089646        76        89

MODEL 122 - TIME PSI
     MONTH       PSI  OBS_PREV  OBS_CURR
0  2022-12  0.410690        58        55
1  2023-06  0.460844        90        67
2  2023-07  0.149157        67        69
3  2023-08  0.080110        69       172
4  2023-09  0.192904       172       120
5  2023-10  0.279798       120        53

MODEL 184 - TIME PSI
      MONTH       PSI  OBS_PREV  OBS_CURR
0   2023-02  0.096341       152       394
1   2023-03  0.113681       394       117
2   2023-06  0.179888       111       283
3   2023-07  0.478001       2

In [ ]:
# SCORING_ID = 42
# Vaqt bo‘yicha PSI tahlili model taqsimotining yuqori darajada o‘zgaruvchan ekanligini ko‘rsatadi. Bir nechta davrlarda PSI 0.5 dan, ayrim oylarda 
# esa 1.0 va hatto 3.4 dan yuqori qiymatlarni olgan. Kuzatilgan keskin o‘zgarishlar modelning vaqt bo‘yicha stabil emasligini ko‘rsatadi.
# 
# SCORING_ID = 122
# Modelda PSI ko‘rsatkichlari ko‘p hollarda 0.1–0.5 oralig‘ida shakllangan. Ayrim davrlarda 0.4 dan yuqori qiymatlar kuzatilgan bo‘lib, bu 
# portfel tarkibida sezilarli o‘zgarishlar mavjudligini bildiradi. Umumiy baholashda model stabiligi o‘rtacha-past darajada.
# 
# SCORING_ID = 184
# Modelda PSI qiymatlari tez-tez 0.25 dan yuqori va bir qator davrlarda 0.5–1.7 oralig‘iga chiqadi. Vaqt davomida score taqsimotining sezilarli 
# siljishi kuzatilgan. Modelning stabiligi qoniqarli emas va drift riski yuqori.
# 
# SCORING_ID = 343
# Ayrim davrlarda PSI juda past bo‘lsa-da, bir nechta keskin sakrashlar (1.1 va 6.88 gacha) aniqlangan. Bu model portfeli tarkibida katta 
# strukturaviy o‘zgarishlar bo‘lganligini ko‘rsatadi. Model vaqt bo‘yicha barqaror emas.
# 
# SCORING_ID = 504
# PSI ko‘rsatkichlari aksariyat davrlarda 0.2–0.9 oralig‘ida shakllangan va 2.23 gacha yetgan holatlar mavjud. Bu score taqsimotida sezilarli 
# drift mavjudligini ko‘rsatadi. Model stabiligi past baholanadi.
# 
# SCORING_ID = 727
# Model eng barqaror natijalarni namoyish etgan. PSI qiymatlari uzoq vaqt davomida 0.01 dan ham past darajada saqlangan. Faqat oxirgi davrlarda 
# kuzatuvlar sonining kamayishi fonida ayrim oylarda 0.2–0.55 gacha oshish kuzatiladi. Shunga qaramay, boshqa modellarga nisbatan vaqt 
# bo‘yicha eng yuqori stabilik ushbu modelda kuzatilgan.

In [16]:
def make_lift_table(df_model, target, cutoff_date, n_bins=10):

    tmp = df_model.loc[
        df_model['APPLICATION_DATE'] <= cutoff_date,
        [target, 'SCORE_BALL']
    ].dropna().copy()

    if len(tmp) < 100 or tmp[target].nunique() < 2:
        return None

    # =========================
    # DECILE (HIGH SCORE = BETTER ASSUMPTION FIXED)
    # =========================
    tmp['DECILE'] = pd.qcut(
        tmp['SCORE_BALL'],
        q=n_bins,
        labels=False,
        duplicates='drop'
    ) + 1

    lift = (
        tmp.groupby('DECILE')
        .agg(
            cnt=(target, 'count'),
            bads=(target, 'sum'),
            bad_rate=(target, 'mean')
        )
        .reset_index()
        .sort_values('DECILE')
    )

    lift['bad_rate'] = lift['bad_rate'] * 100

    overall_bad_rate = tmp[target].mean()

    # =========================
    # LIFT (FIXED + SAFE)
    # =========================
    lift['lift'] = lift['bad_rate'] / (overall_bad_rate * 100)

    # normalize again for stability
    lift['lift'] = lift['lift'].replace([np.inf, -np.inf], np.nan)

    return lift, overall_bad_rate
all_lift = {}

for scoring_id, df_model in df.groupby('SCORING_ID'):

    print("\n" + "="*80)
    print(f"MODEL: {scoring_id}")
    print("="*80)

    for target, maturity_days in targets.items():

        cutoff_date = max_date - pd.Timedelta(days=maturity_days)

        result = make_lift_table(df_model, target, cutoff_date)

        if result is None:
            print(f"{target}: data yo'q")
            continue

        lift_df, overall_br = result

        all_lift[(scoring_id, target)] = lift_df

        print("\n" + "-"*60)
        print(f"{target} | Overall Bad Rate = {overall_br:.4f}")
        print("-"*60)

        print(lift_df)

        # =========================
        # SAFE TOP LIFT
        # =========================
        if 1 in lift_df['DECILE'].values:
            top_lift = lift_df.loc[lift_df['DECILE'] == 1, 'lift'].values[0]
        else:
            top_lift = lift_df['lift'].max()

        print(f"\nTOP LIFT (worst decile): {top_lift:.2f}x")


MODEL: 42

------------------------------------------------------------
TARGET_30 | Overall Bad Rate = 0.2111
------------------------------------------------------------
   DECILE  cnt  bads   bad_rate      lift
0       1  272  81.0  29.779412  1.410519
1       2  350  73.0  20.857143  0.987911
2       3  211  52.0  24.644550  1.167304
3       4  282  60.0  21.276596  1.007779
4       5  233  41.0  17.596567  0.833472
5       6  269  47.0  17.472119  0.827577
6       7  253  56.0  22.134387  1.048408
7       8  280  66.0  23.571429  1.116475
8       9  254  52.0  20.472441  0.969689
9      10  239  30.0  12.552301  0.594547

TOP LIFT (worst decile): 1.41x

------------------------------------------------------------
TARGET_60 | Overall Bad Rate = 0.0692
------------------------------------------------------------
   DECILE  cnt  bads   bad_rate      lift
0       1  272  31.0  11.397059  1.646034
1       2  350  27.0   7.714286  1.114145
2       3  211  18.0   8.530806  1.232072
3    

In [ ]:
# Modellar bo‘yicha lift xulosasi
# Model 343
# Lift ≈ 1.0x
# Eng yomon holat: decilelar deyarli farqlanmagan
# Xulosa: Model riskni umuman ajrata olmaydi (randomga yaqin). Ishlatishga yaroqsiz.
# 
# Model 122
# Lift ≈ 0.5x – 1.3x
# Ba’zi targetlarda hatto 1.0 dan past
# Xulosa: Ranking noto‘g‘ri ishlayapti, model signal bermaydi yoki teskari ishlash holatlari bor.
# 
# Model 184
# Lift ≈ 1.2x – 1.3x
# Past–o‘rtacha ajratish qobiliyati
# Xulosa: Minimal biznes qiymat bor, lekin kuchsiz model. Faqat support-level model sifatida qolishi mumkin.
# 
# Model 42
# Lift ≈ 1.4x – 1.6x
# O‘rtacha past segmentatsiya
# Xulosa: Riskni qisman ajratadi, lekin kuchli qarorlar uchun yetarli emas.
# 
# Model 504
# Lift ≈ 1.4x – 1.6x
# Segmentatsiya mavjud, lekin noturg‘un (PSI yuqori)
# Xulosa: Ajratish bor, lekin stability muammo. Monitoring shart.
# 
# Model 727
# Lift ≈ 2.5x – 2.8x
# Juda kuchli separation
# Xulosa: Eng yaxshi model. Top risk segment aniq ajralgan. Production uchun mos.

In [17]:
def time_based_metrics(df_model, target, score_col='SCORE_BALL', date_col='APPLICATION_DATE'):

    df_model = df_model.copy()
    df_model[date_col] = pd.to_datetime(df_model[date_col])

    # =========================
    # SAFE MONTH FORMAT (KEEP PERIOD, NOT STRING)
    # =========================
    df_model['MONTH'] = df_model[date_col].dt.to_period('M')

    results = []

    months = sorted(df_model['MONTH'].unique())

    for month in months:

        tmp = df_model[df_model['MONTH'] == month].dropna(subset=[target, score_col])

        # =========================
        # STRONG SAMPLE FILTER
        # =========================
        if len(tmp) < 50 or tmp[target].nunique() < 2:
            continue

        # =========================
        # AUC / GINI (CONSISTENT)
        # =========================
        auc = roc_auc_score(tmp[target], -tmp[score_col])
        gini = 2 * auc - 1

        # =========================
        # KS (CONSISTENT SIGN)
        # =========================
        ks = ks_2samp(
            -tmp.loc[tmp[target] == 0, score_col],
            -tmp.loc[tmp[target] == 1, score_col]
        ).statistic

        results.append({
            'MONTH': month,   # keep PERIOD (IMPORTANT FIX)
            'OBS': len(tmp),
            'BAD_RATE': tmp[target].mean(),
            'AUC': auc,
            'GINI': gini,
            'KS': ks
        })

    return pd.DataFrame(results)


all_time_metrics = {}

for scoring_id, df_model in df.groupby('SCORING_ID'):

    df_model = df_model.copy()
    df_model['APPLICATION_DATE'] = pd.to_datetime(df_model['APPLICATION_DATE'])

    print("\n" + "="*90)
    print(f"MODEL: {scoring_id}")
    print("="*90)

    for target, maturity_days in targets.items():

        cutoff_date = max_date - pd.Timedelta(days=maturity_days)

        tmp_model = df_model[df_model['APPLICATION_DATE'] <= cutoff_date]

        if len(tmp_model) < 100:
            print(f"\n{target}: insufficient data")
            continue

        time_df = time_based_metrics(tmp_model, target)

        if time_df.empty:
            print(f"\n{target}: no valid monthly segments")
            continue

        all_time_metrics[(scoring_id, target)] = time_df

        print("\n" + "-"*70)
        print(f"{target} | OBS = {len(tmp_model):,}")
        print("-"*70)

        print(time_df)


MODEL: 42

----------------------------------------------------------------------
TARGET_30 | OBS = 2,643
----------------------------------------------------------------------
      MONTH  OBS  BAD_RATE       AUC      GINI        KS
0   2022-08  166  0.253012  0.689324  0.378648  0.337174
1   2022-09  255  0.227451  0.586863  0.173727  0.150971
2   2022-10  257  0.210117  0.653758  0.307517  0.254242
3   2022-11  387  0.183463  0.593978  0.187957  0.152656
4   2022-12  342  0.213450  0.482864 -0.034272  0.102460
5   2023-01   87  0.218391  0.541022  0.082043  0.175697
6   2023-02  109  0.256881  0.597222  0.194444  0.176808
7   2023-03   82  0.219512  0.502170  0.004340  0.140625
8   2023-05   59  0.220339  0.612876  0.225753  0.314381
9   2023-08  275  0.214545  0.558027  0.116055  0.128923
10  2023-09   61  0.131148  0.472877 -0.054245  0.297170
11  2024-04   64  0.234375  0.653061  0.306122  0.248980
12  2024-05   76  0.223684  0.666500  0.333001  0.276171
13  2024-06   89  0.2134

In [2]:
# Model 727
# Juda kuchli model, yuqori lift va stabil signal
# TARGET 30: AUC ~0.71–0.78 → yuqori va stabil
# TARGET 60: AUC ~0.69–0.77 → juda barqaror
# TARGET 90: AUC ~0.69–0.78 → eng stabil segment
# Kuzatuv:
# 2026 boshida TARGET 30 da keskin spike (AUC 0.91) va volatility kuzatildi
# OBS kichik oylar natijani distort qilgan bo‘lishi mumkin
# Xulosa:
# Model kuchli, lekin recent periodda small-sample instability signali bor
# 
# Model 504
# Umumiy baho: Yaxshi, lekin juda volatile model
# TARGET 30: AUC 0.47–0.74 → kuchli tebranish
# TARGET 60: AUC 0.50–0.77 → unstable
# TARGET 90: AUC 0.50–0.82 → ba’zan juda kuchli spike
# Kuzatuv:
# Oylik performance “wave pattern” ko‘rsatadi
# Ba’zi davrlarda keskin drop (0.47 atrofida)
# Xulosa:
# Modelda drift + instability mavjud, lekin signal hali saqlangan
# 
# Model 184
# Umumiy baho: O‘rtacha, seasonal drift bor
# TARGET 30: AUC 0.54–0.69 → o‘rtacha
# TARGET 60: AUC 0.50–0.69 → stabilroq
# TARGET 90: AUC 0.49–0.66 → nisbatan yaxshi
# Kuzatuv:
# Ba’zi oylarda drop (2024-05, 2024-12)
# Ba’zi oylarda recovery (2024-08, 2025-09)
# Xulosa:
# Model ishlaydi, lekin seasonality va drift ta’siri bor
# 
# Model 122
# Umumiy baho: Zaif va beqaror model
# TARGET 30: AUC 0.38–0.73 → inconsistent
# TARGET 60: AUC 0.40–0.71 → unstable
# TARGET 90: AUC 0.35–0.65 → weak signal
# Kuzatuv:
# Ba’zi oylarda yaxshi, lekin tez buziladi
# Negative / near-random periods mavjud
# Xulosa:
# Modelda barqaror predictive power yo‘q
# 
# Model 343
# Umumiy baho: Juda kuchsiz model (near-random)
# TARGET 30: AUC ~0.49–0.55
# TARGET 60: AUC ~0.49–0.55
# TARGET 90: AUC ~0.49–0.55
# Kuzatuv:
# KS juda past
# AUC deyarli random (0.5 atrofida)
# Hech qanday stabil signal yo‘q
# 
# Xulosa:
# Model predictive value bermaydi

In [18]:
def stability_score(df_time):

    df_time = df_time.copy()

    # =========================
    # SAFETY CHECK
    # =========================
    if len(df_time) < 3:
        return {
            'GINI_VOLATILITY': np.nan,
            'KS_VOLATILITY': np.nan,
            'GINI_CV': np.nan,
            'KS_CV': np.nan
        }

    # =========================
    # STATS
    # =========================
    gini_mean = df_time['GINI'].mean()
    ks_mean = df_time['KS'].mean()

    gini_std = df_time['GINI'].std()
    ks_std = df_time['KS'].std()

    return {
        'GINI_VOLATILITY': gini_std,
        'KS_VOLATILITY': ks_std,
        'GINI_CV': gini_std / (abs(gini_mean) + 1e-6),
        'KS_CV': ks_std / (abs(ks_mean) + 1e-6)
    }
time_summary = []

for key, df_time in all_time_metrics.items():

    scoring_id, target = key

    if len(df_time) < 2:
        continue

    stats = stability_score(df_time)

    time_summary.append({
        'SCORING_ID': scoring_id,
        'TARGET': target,

        # =========================
        # LEVEL
        # =========================
        'AVG_GINI': df_time['GINI'].mean(),
        'STD_GINI': df_time['GINI'].std(),

        'AVG_KS': df_time['KS'].mean(),
        'STD_KS': df_time['KS'].std(),

        # =========================
        # STABILITY (NEW)
        # =========================
        'GINI_VOLATILITY': stats['GINI_VOLATILITY'],
        'KS_VOLATILITY': stats['KS_VOLATILITY'],
        'GINI_CV': stats['GINI_CV'],
        'KS_CV': stats['KS_CV'],

        # =========================
        # EXTRA INSIGHT (OPTIONAL BUT USEFUL)
        # =========================
        'GINI_MIN': df_time['GINI'].min(),
        'GINI_MAX': df_time['GINI'].max(),
        'KS_MIN': df_time['KS'].min(),
        'KS_MAX': df_time['KS'].max()
    })

time_summary_df = pd.DataFrame(time_summary)
time_summary_df

,SCORING_ID,TARGET,AVG_GINI,STD_GINI,AVG_KS,STD_KS,GINI_VOLATILITY,KS_VOLATILITY,GINI_CV,KS_CV,GINI_MIN,GINI_MAX,KS_MIN,KS_MAX
0,42,TARGET_30,0.187489,0.149884,0.225984,0.091971,0.149884,0.091971,0.799426,0.406977,-0.054245,0.403759,0.102460,0.407519
1,42,TARGET_60,0.324916,0.266342,0.411121,0.213827,0.266342,0.213827,0.819722,0.520107,-0.057255,0.850704,0.140857,0.859155
2,42,TARGET_90,0.243062,0.359935,0.460169,0.206549,0.359935,0.206549,1.480827,0.448853,-0.531447,0.844749,0.194602,0.876712
3,122,TARGET_30,0.095105,0.175478,0.234902,0.099286,0.175478,0.099286,1.845074,0.422670,-0.235192,0.464387,0.152381,0.457265
4,122,TARGET_60,0.123722,0.204031,0.295567,0.118536,0.204031,0.118536,1.649095,0.401044,-0.193878,0.431034,0.172414,0.490566
5,122,TARGET_90,0.108456,0.210288,0.309396,0.120981,0.210288,0.120981,1.938906,0.391021,-0.282051,0.467742,0.173729,0.486175
6,184,TARGET_30,0.230271,0.128947,0.236068,0.104741,0.128947,0.104741,0.559976,0.443689,-0.033305,0.503268,0.087651,0.529412
7,184,TARGET_60,0.267050,0.150623,0.284399,0.159844,0.150623,0.159844,0.564023,0.562038,-0.038796,0.645161,0.059358,0.790323
8,184,TARGET_90,0.303787,0.228378,0.334255,0.206568,0.228378,0.206568,0.751769,0.617992,-0.078403,0.875000,0.087798,0.902174
9,343,TARGET_30,0.068979,0.066415,0.064772,0.049425,0.066415,0.049425,0.962818,0.763054,-0.020104,0.262424,0.017303,0.221212


In [19]:
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from scipy.stats import ks_2samp

final_summary = []

for scoring_id, df_model in df.groupby('SCORING_ID'):

    df_model = df_model.copy()
    df_model['APPLICATION_DATE'] = pd.to_datetime(df_model['APPLICATION_DATE'])

    for target, maturity_days in targets.items():

        cutoff_date = max_date - pd.Timedelta(days=maturity_days)

        tmp_model = df_model[df_model['APPLICATION_DATE'] <= cutoff_date]

        if len(tmp_model) < 100 or tmp_model[target].nunique() < 2:
            continue

        # =========================
        # AUC / GINI
        # =========================
        auc = roc_auc_score(tmp_model[target], -tmp_model['SCORE_BALL'])
        gini = 2 * auc - 1

        # =========================
        # KS (CONSISTENT SIGN)
        # =========================
        ks = ks_2samp(
            -tmp_model.loc[tmp_model[target] == 0, 'SCORE_BALL'],
            -tmp_model.loc[tmp_model[target] == 1, 'SCORE_BALL']
        ).statistic

        # =========================
        # PSI (SAFE + NO OVERLAP BIAS FIX)
        # =========================
        dates = np.sort(tmp_model['APPLICATION_DATE'].unique())

        if len(dates) > 5:
            split_date = dates[int(len(dates) * 0.7)]

            dev_df = tmp_model[tmp_model['APPLICATION_DATE'] <= split_date]
            oot_df = tmp_model[tmp_model['APPLICATION_DATE'] > split_date]

            psi = calculate_psi(
                dev_df['SCORE_BALL'].dropna(),
                oot_df['SCORE_BALL'].dropna()
            ) if len(dev_df) > 50 and len(oot_df) > 50 else np.nan
        else:
            psi = np.nan

        # =========================
        # LIFT (FIXED DECILE SAFETY)
        # =========================
        tmp = tmp_model[[target, 'SCORE_BALL']].dropna().copy()

        tmp['DECILE'] = pd.qcut(
            tmp['SCORE_BALL'],
            q=10,
            labels=False,
            duplicates='drop'
        ) + 1

        lift_table = (
            tmp.groupby('DECILE')[target].mean().sort_index()
        )

        overall_br = tmp[target].mean()

        # FIX: explicit worst decile assumption
        top_lift = (lift_table.iloc[0] / overall_br) if overall_br > 0 else np.nan

        # =========================
        # STORE
        # =========================
        final_summary.append({
            'SCORING_ID': scoring_id,
            'TARGET': target,

            'OBS': len(tmp_model),

            # performance
            'AUC': auc,
            'GINI': gini,
            'KS': ks,

            # stability
            'PSI': psi,

            # ranking power
            'TOP_LIFT': top_lift,

        })

def risk_flag(row):

    # =========================
    # HARD FAIL CONDITIONS
    # =========================
    if (row['GINI'] < 0.2) or (row['KS'] < 0.1):
        return 'BAD'

    # =========================
    # WARNING CONDITIONS (MORE REALISTIC)
    # =========================
    if (
        (row['PSI'] > 0.25) 
    ):
        return 'WARNING'

    return 'GOOD'


final_df = pd.DataFrame(final_summary)
final_df['STATUS'] = final_df.apply(risk_flag, axis=1)

final_df

,SCORING_ID,TARGET,OBS,AUC,GINI,KS,PSI,TOP_LIFT,STATUS
0,42,TARGET_30,2643,0.545942,0.091884,0.077590,4.624665,1.410519,BAD
1,42,TARGET_60,2643,0.565131,0.130261,0.108563,4.624665,1.646034,BAD
2,42,TARGET_90,2643,0.541962,0.083924,0.078253,4.624665,1.650042,BAD
3,122,TARGET_30,1272,0.533602,0.067205,0.122124,0.458152,0.845502,BAD
4,122,TARGET_60,1272,0.539769,0.079538,0.159923,0.458152,0.764686,BAD
5,122,TARGET_90,1271,0.538424,0.076849,0.148820,0.472882,0.516387,BAD
6,184,TARGET_30,6731,0.567923,0.135846,0.117062,0.688074,1.274479,BAD
7,184,TARGET_60,6731,0.563953,0.127907,0.102537,0.688074,1.294397,BAD
8,184,TARGET_90,6731,0.563919,0.127838,0.096852,0.688074,1.302399,BAD
9,343,TARGET_30,46728,0.516976,0.033951,0.032126,2.402779,1.019762,BAD


In [ ]:
# MODEL 42 bo‘yicha natijalar shuni ko‘rsatadiki, modelning discriminative kuchi past (GINI 0.08–0.13, KS 0.07–0.11). 
# TOP LIFT 1.4–1.65 atrofida bo‘lib, faqat minimal uplift beradi. PSI juda yuqori (~4.62) bo‘lgani uchun vaqt bo‘yicha kuchli drift mavjud. 
# Umuman olganda, model barqaror emas va signal kuchsiz, shuning uchun production uchun xavfli (BAD).
# 
# MODEL 122 da ham xuddi shunga o‘xshash holat kuzatiladi: GINI juda past (0.06–0.08), KS past (0.12–0.16), 
# TOP LIFT esa 1 dan past darajaga tushib ketgan. PSI ~0.46 bo‘lib, o‘rtacha drift mavjud. 
# Model deyarli random behaviorga yaqin va kredit riskni ajratish qobiliyati juda zaif, shu sababli model ishonchsiz (BAD).
# 
# MODEL 184 da biroz yaxshiroq signal mavjud (GINI 0.12–0.13, KS 0.09–0.11), TOP LIFT ~1.27–1.30 atrofida. 
# Biroq PSI ~0.68 bo‘lib, sezilarli drift mavjud. Bu modelda signal bor, lekin vaqt bo‘yicha barqaror emas, shuning uchun o‘rtacha sifatli, 
# lekin riskli model hisoblanadi (BAD).
# 
# MODEL 343 eng zaif model hisoblanadi. GINI juda past (0.01–0.03), KS ham juda past (0.02–0.03), 
# TOP LIFT esa ~1 atrofida, ya’ni deyarli hech qanday ranking kuchi yo‘q. PSI ~2.40 bo‘lib, katta drift ham mavjud. 
# Bu model amalda predictive value bermaydi va random modelga yaqin (BAD).
# 
# MODEL 504 o‘rtacha darajadagi model hisoblanadi. GINI 0.21–0.24, KS 0.15–0.18 bo‘lib, signal mavjudligini ko‘rsatadi. 
# TOP LIFT 1.4–1.57 atrofida bo‘lib, yaxshi uplift beradi. Biroq PSI ~1.48 bo‘lgani uchun drift mavjud. 
# Model ishlatilishi mumkin, lekin doimiy monitoring talab etiladi (WARNING).
# 
# MODEL 727 eng yaxshi model hisoblanadi. GINI yuqori (0.44–0.46), KS 0.33–0.37, TOP LIFT esa 2.4–2.8 atrofida bo‘lib juda kuchli ranking power beradi. 
# PSI esa juda past (0.006–0.11), ya’ni model juda stabil. Bu model ham signal, ham stability bo‘yicha eng yaxshi natija ko‘rsatgan va 
# production uchun to‘liq tayyor (GOOD).